In [1]:
%cd '/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series'

/Users/lorenzodelcol/Desktop/RA/REPLICATION DROPBOX (NEW)/Raw data UPDATE/Updated 2026 Series


/opt/anaconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import pandas as pd
import numpy as np
import openpyxl

## PHASE A ans B
Pahse A: Computing the daily returns

Phase B: Computing the montly standard deviation of the daily returns from Phase A

In [5]:
# 1. Create a list of the 3 raw files, along with the names we want for the output files
datasets = [
    {
        "input": "No_DS_S-MKT_RETURNS_DEL_COL_Updated_2026.xlsx",
        "out_daily": "PHASE_A_MKT_daily_df.xlsx",
        "out_monthly": "PHASE_B_MKT_monthly_df.xlsx"
    },
    {
        "input": "No_DS_10YR_RETURNS_DEL_COL_update2026.xlsx",
        "out_daily": "PHASE_A_10YR_daily_df.xlsx",
        "out_monthly": "PHASE_B_10YR_monthly_df.xlsx"
    },
    {
        "input": "No_DS_EX_RETURNS_DEL_COL_Updated 2026.xlsx",
        "out_daily": "PHASE_A_EX_daily_df.xlsx",
        "out_monthly": "PHASE_B_EX_monthly_df.xlsx"
    }
]

# 2. Start the Master Loop
for data in datasets:
    file_path = data["input"]
    print(f"\n{'='*50}\n Starting to process: {file_path}")
    
    # Load the workbook
    all_sheets = pd.read_excel(file_path, sheet_name=None)
    daily_df = pd.DataFrame()

    # ==========================================
    # PHASE A: Calculate Daily Returns
    # ==========================================
    print("   -> Extracting raw data and computing daily returns...")
    for country_name, df in all_sheets.items():
        try:
            # Grab Raw Data from Columns A and B, starting at Excel Row 8
            df_raw = df.iloc[7:, [0, 1]].copy()
            df_raw.columns = ["Date", "Raw_Value"]
            
            df_raw["Date"] = pd.to_datetime(df_raw["Date"], errors="coerce")
            df_raw["Raw_Value"] = pd.to_numeric(df_raw["Raw_Value"], errors="coerce")
            df_raw = df_raw.dropna()
            
            # Calculate Return
            df_raw[country_name] = df_raw["Raw_Value"].pct_change() * 100
            df_raw.set_index('Date', inplace=True)
            df_clean = df_raw[[country_name]].dropna()
            
            if daily_df.empty:
                daily_df = df_clean
            else:
                daily_df = daily_df.join(df_clean, how='outer')
                
        except Exception as e:
            print(f"  Could not process {country_name}. Error: {e}")

    daily_df = daily_df.sort_index()

    print("   -> Cleaning infinite values caused by Zero Lower Bound...")
    daily_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # ==========================================
    # PHASE B: Calculate Monthly SD and SK
    # ==========================================
    print("   -> Grouping into months and computing SD & SK...")
    monthly_df = daily_df.resample('MS').agg(['std', 'skew'])

    # Flatten the columns
    new_columns = []
    for country, stat in monthly_df.columns:
        if stat == 'std':
            new_columns.append(f'SD {country}')
        elif stat == 'skew':
            new_columns.append(f'SK {country}')
            
    monthly_df.columns = new_columns

    # ==========================================
    # SAVE TO EXCEL
    # ==========================================
    daily_df.index = daily_df.index.strftime('%Y-%m-%d')
    monthly_df.index = monthly_df.index.strftime('%Y-%m-%d')
    
    daily_df.to_excel(data["out_daily"])
    monthly_df.to_excel(data["out_monthly"])
    
    print(f" Complete! Saved daily and monthly files for this dataset.")

print(f"\n{'='*50}\n ALL 3 DATASETS HAVE BEEN SUCCESSFULLY PROCESSED!")


 Starting to process: NO_DS_S-MKT_RETURNS_DEL_COL_Updated_2026.xlsx
   -> Extracting raw data and computing daily returns...
   -> Cleaning infinite values caused by Zero Lower Bound...
   -> Grouping into months and computing SD & SK...
 Complete! Saved daily and monthly files for this dataset.

 Starting to process: No_DS_10YR_RETURNS_DEL_COL_update2026.xlsx
   -> Extracting raw data and computing daily returns...
   -> Cleaning infinite values caused by Zero Lower Bound...
   -> Grouping into months and computing SD & SK...
 Complete! Saved daily and monthly files for this dataset.

 Starting to process: No_DS_EX_RETURNS_DEL_COL_Updated 2026.xlsx
   -> Extracting raw data and computing daily returns...
   -> Cleaning infinite values caused by Zero Lower Bound...
   -> Grouping into months and computing SD & SK...
 Complete! Saved daily and monthly files for this dataset.

 ALL 3 DATASETS HAVE BEEN SUCCESSFULLY PROCESSED!
